# Week 8 Implementation: Multi-Step RLM with Training Loop + LLM Agent

**Progress since Week 4:**
- Added multi-step tasks (KV extraction, aggregation) requiring 2+ REPL steps
- Built HeuristicMultiStepAgent with strategy selection
- Implemented reward function: R = C - λ_s(steps/max_steps) - λ_t(tokens)
- Created training loop skeleton for trajectory collection and imitation learning
- **NEW (Week 8):** Added LLMAgent using HuggingFace Inference API (Qwen2.5-Coder-7B-Instruct)

This notebook demonstrates the full pipeline end-to-end, including LLM-based code generation.

## 1. Setup and Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from src.utils import (
    generate_task,
    generate_kv_extraction_task,
    generate_multistep_task,
    compute_reward,
    safe_execute_code,
)
from src.models import (
    DeterministicAgent,
    HeuristicMultiStepAgent,
    LLMAgent,
    EvaluationFramework,
    TrainingLoop,
)

print("All imports successful.")

# Check if HF_TOKEN is available for LLM demos
HF_TOKEN_SET = bool(os.environ.get("HF_TOKEN"))
if HF_TOKEN_SET:
    print("HF_TOKEN found — LLM agent demos will run.")
else:
    print("HF_TOKEN not set — LLM agent demos will be skipped.")
    print("Set it with: export HF_TOKEN=your_token_here")

## 2. Problem Formulation

### Objective
Maximize expected reward under policy π:

$$\max_\pi \; E\left[ R \mid \pi \right] \quad \text{where} \quad R = C - \lambda_s \frac{T}{T_{\max}} - \lambda_t \cdot N_{\text{tokens}}$$

- $C \in \{0, 1\}$: correctness of final answer
- $T$: number of REPL steps taken
- $T_{\max}$: maximum allowed steps
- $N_{\text{tokens}}$: total tokens consumed
- $\lambda_s = 0.05$, $\lambda_t = 0.0001$: penalty weights

### Task Types

| Task | Steps | Description |
|------|-------|-------------|
| Needle-in-haystack | 1 | Find KEY=VALUE in random text |
| KV extraction | 2 | Filter log lines by field, extract target |
| Aggregation | 2 | Find all METRIC_* keys, sum values |

## 3. Task Examples

### 3.1 Simple Needle Task (Week 4)

In [ ]:
haystack, question, answer = generate_task(num_sentences=10, num_needles=1)
print(f"Question: {question}")
print(f"Answer: {answer}")
print(f"Haystack preview: {haystack[:200]}...")

### 3.2 KV Extraction Task (Week 6)

In [ ]:
context, question, answer = generate_kv_extraction_task(num_entries=15)
print(f"Question: {question}")
print(f"Answer: {answer}")
print(f"\nLog preview (first 5 lines):")
for line in context.split('\n')[:5]:
    print(f"  {line}")

### 3.3 Aggregation Task (Week 6)

In [ ]:
context, question, answer = generate_multistep_task(num_sentences=10, num_keys=3)
print(f"Question: {question}")
print(f"Answer: {answer}")
print(f"\nContext preview: {context[:300]}...")

## 4. Agent Comparison

### 4.1 Deterministic Agent on Simple Tasks

In [ ]:
det_agent = DeterministicAgent()
eval_simple = EvaluationFramework(
    agents=[det_agent],
    task_generator=lambda: generate_task(num_sentences=15, num_needles=1),
    num_episodes=10,
)
eval_simple.run_evaluation()
eval_simple.display_results()

### 4.2 HeuristicMultiStepAgent on All Task Types

In [ ]:
import random

def mixed_task_generator():
    """Randomly select from all three task types."""
    choice = random.choice(['needle', 'kv', 'aggregate'])
    if choice == 'needle':
        return generate_task(num_sentences=15, num_needles=1)
    elif choice == 'kv':
        return generate_kv_extraction_task(num_entries=20)
    else:
        return generate_multistep_task(num_sentences=15, num_keys=3)

multi_agent = HeuristicMultiStepAgent()
eval_multi = EvaluationFramework(
    agents=[multi_agent],
    task_generator=mixed_task_generator,
    num_episodes=15,
)
eval_multi.run_evaluation()
eval_multi.display_results()

### 4.3 Side-by-Side Comparison

In [ ]:
# Compare both agents on simple needle tasks
det = DeterministicAgent()
heuristic = HeuristicMultiStepAgent()

eval_compare = EvaluationFramework(
    agents=[det, heuristic],
    task_generator=lambda: generate_task(num_sentences=15, num_needles=1),
    num_episodes=10,
)
eval_compare.run_evaluation()
eval_compare.display_results()

## 5. Reward System

The reward function balances correctness against efficiency:

$$R = C - 0.05 \cdot \frac{T}{T_{\max}} - 0.0001 \cdot N_{\text{tokens}}$$

In [ ]:
# Demonstrate reward computation across different scenarios
scenarios = [
    ("Correct, 1 step",    True,  1,  10),
    ("Correct, 5 steps",   True,  5,  10),
    ("Correct, 10 steps",  True,  10, 10),
    ("Wrong, 1 step",      False, 1,  10),
    ("Wrong, 10 steps",    False, 10, 10),
]

print(f"{'Scenario':<22} {'Reward':>8}")
print("-" * 32)
for label, correct, steps, max_s in scenarios:
    r = compute_reward(is_correct=correct, num_steps=steps, max_steps=max_s)
    print(f"{label:<22} {r:>8.3f}")

## 6. Training Loop

The training loop collects trajectories and filters successful ones for imitation learning.

**Current status:** Trajectory collection and statistics work. Actual weight updates (behavior cloning on successful rollouts, then reward-weighted policy gradients) are planned for after LLM integration.

In [ ]:
# Run training loop with deterministic agent on simple tasks
trainer = TrainingLoop(
    agent=DeterministicAgent(),
    task_generator=lambda: generate_task(num_sentences=15, num_needles=1),
    batch_size=8,
)
history = trainer.run(num_iterations=5)

print(f"\nTrajectory buffer: {len(trainer.trajectory_buffer)} episodes")
print(f"Successful trajectories: {len(trainer.filter_successes())}")

In [ ]:
# Training loop with multi-step agent on mixed tasks
trainer_multi = TrainingLoop(
    agent=HeuristicMultiStepAgent(),
    task_generator=mixed_task_generator,
    batch_size=8,
)
history_multi = trainer_multi.run(num_iterations=5)

print(f"\nTrajectory buffer: {len(trainer_multi.trajectory_buffer)} episodes")
successes = trainer_multi.filter_successes()
print(f"Successful trajectories: {len(successes)}")
if successes:
    print(f"Sample successful action: {successes[0].actions[0][:80]}...")

## 7. Training Statistics Visualization

In [ ]:
# Plot training history
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    iters = range(1, len(history_multi) + 1)
    accs = [h['accuracy'] for h in history_multi]
    rewards = [h['avg_reward'] for h in history_multi]
    steps = [h['avg_steps'] for h in history_multi]

    axes[0].plot(iters, accs, 'b-o')
    axes[0].set_title('Accuracy per Iteration')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_ylim(0, 1.1)

    axes[1].plot(iters, rewards, 'g-o')
    axes[1].set_title('Avg Reward per Iteration')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Reward')

    axes[2].plot(iters, steps, 'r-o')
    axes[2].set_title('Avg REPL Steps per Iteration')
    axes[2].set_xlabel('Iteration')
    axes[2].set_ylabel('Steps')

    plt.tight_layout()
    plt.savefig('../figures/training_stats.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Figure saved to figures/training_stats.png")
except ImportError:
    print("matplotlib not available; skipping plots.")
    print("Training stats:")
    for i, h in enumerate(history_multi):
        print(f"  Iter {i+1}: acc={h['accuracy']:.1%} reward={h['avg_reward']:.3f} steps={h['avg_steps']:.1f}")

## 8. Edge Case Testing

In [ ]:
agent = HeuristicMultiStepAgent()

# Edge case 1: Missing needle
print("=== Missing Needle ===")
h, q, a = generate_task(num_sentences=10, num_needles=0)
pred, tr = agent.run_episode(h, q, a)
print(f"  Expected: {a}  Got: {pred}  Pass: {pred == a or 'not found' in pred.lower()}")

# Edge case 2: Very long haystack
print("\n=== Long Haystack (50 sentences) ===")
h, q, a = generate_task(num_sentences=50, num_needles=1)
pred, tr = agent.run_episode(h, q, a)
print(f"  Haystack: {len(h)} chars  Expected: {a}  Got: {pred}  Pass: {pred == a}")

# Edge case 3: Large log file
print("\n=== Large KV Log (100 entries) ===")
ctx, q, a = generate_kv_extraction_task(num_entries=100)
pred, tr = agent.run_episode(ctx, q, a)
print(f"  Context: {len(ctx)} chars  Steps: {len(tr)}  Pass: {pred == a}")

# Edge case 4: Many metrics to aggregate
print("\n=== Many Metrics (5 keys) ===")
ctx, q, a = generate_multistep_task(num_sentences=20, num_keys=5)
pred, tr = agent.run_episode(ctx, q, a)
print(f"  Expected sum: {a}  Got: {pred}  Steps: {len(tr)}  Pass: {pred == a}")

## 9. LLM Agent (Week 8)

The `LLMAgent` uses the HuggingFace Inference API to generate Python code for REPL execution. It:
1. Sends a system prompt + context preview + question to the LLM
2. Extracts Python code from the LLM's markdown response
3. Executes in the sandbox with `CONTEXT` variable holding the full text
4. If errors occur, sends them back to the LLM for retry (up to 5 steps)

**Model:** `Qwen/Qwen2.5-Coder-7B-Instruct` (configurable per instance)

**Requires:** `HF_TOKEN` environment variable (free at huggingface.co/settings/tokens)

## 12. Current Limitations and Next Steps

### What Works
- Deterministic baseline: 100% accuracy on needle tasks (1 step)
- Heuristic multi-step agent: handles KV extraction and aggregation (2 steps)
- **LLM agent: generates Python code via HuggingFace API with multi-step retry**
- Reward computation correctly penalises extra steps
- Training loop collects and filters trajectories
- Safe sandbox prevents dangerous code execution

### Current Limitations
- **LLM accuracy varies** — depends on model quality, prompt engineering, and task complexity
- **Rate limits** — free-tier HuggingFace API has request limits; exponential backoff helps
- **Training loop doesn't update weights** — collects trajectories but no gradient updates
- **Limited task diversity** — 3 task types, all synthetic

### Next Steps
1. **Behavior cloning**: Train on successful trajectories from heuristic + LLM agents
2. **Reward-weighted updates**: Move from imitation to RL-style policy gradient
3. **More complex tasks**: Log parsing with multiple filter conditions, transform-then-answer
4. **Token tracking**: Measure and optimise token usage per episode
5. **Model comparison**: Evaluate different HuggingFace models (Qwen 1.5B vs 7B, Mistral, etc.)

### 9.1 LLM Agent on Needle Task

In [ ]:
if HF_TOKEN_SET:
    llm_agent = LLMAgent(max_steps=5)
    haystack, question, answer = generate_task(num_sentences=10, num_needles=1)

    print(f"Question: {question}")
    print(f"Expected: {answer}\n")

    predicted, transcript = llm_agent.run_episode(haystack, question, answer)
    print(f"Predicted: {predicted}")
    print(f"Correct: {predicted == answer}")
    print(f"Steps used: {len(transcript)}")

    # Show what the LLM generated
    for entry in transcript:
        print(f"\n--- Step {entry['step']}: {entry['action']} ---")
        if 'code' in entry:
            print(f"Code:\n{entry['code']}")
        if 'exec_result' in entry:
            er = entry['exec_result']
            print(f"OK: {er['ok']}  stdout: {er['stdout'].strip()[:100]}")
else:
    print("Skipped — set HF_TOKEN to run LLM demos")

### 9.2 LLM Agent on KV Extraction Task

In [ ]:
if HF_TOKEN_SET:
    context, question, answer = generate_kv_extraction_task(num_entries=15)

    print(f"Question: {question}")
    print(f"Expected: {answer}\n")

    llm_agent = LLMAgent(max_steps=5)
    predicted, transcript = llm_agent.run_episode(context, question, answer)
    print(f"Predicted: {predicted}")
    print(f"Correct: {predicted == answer}")
    print(f"Steps used: {len(transcript)}")

    for entry in transcript:
        print(f"\n--- Step {entry['step']}: {entry['action']} ---")
        if 'code' in entry:
            print(f"Code:\n{entry['code']}")
        if 'exec_result' in entry:
            er = entry['exec_result']
            print(f"OK: {er['ok']}  stdout: {er['stdout'].strip()[:100]}")
            if er['stderr']:
                print(f"stderr: {er['stderr'].strip()[:200]}")
else:
    print("Skipped — set HF_TOKEN to run LLM demos")

### 9.3 LLM Agent on Aggregation Task

In [ ]:
if HF_TOKEN_SET:
    context, question, answer = generate_multistep_task(num_sentences=10, num_keys=3)

    print(f"Question: {question}")
    print(f"Expected: {answer}\n")

    llm_agent = LLMAgent(max_steps=5)
    predicted, transcript = llm_agent.run_episode(context, question, answer)
    print(f"Predicted: {predicted}")
    print(f"Correct: {predicted == answer}")
    print(f"Steps used: {len(transcript)}")

    for entry in transcript:
        print(f"\n--- Step {entry['step']}: {entry['action']} ---")
        if 'code' in entry:
            print(f"Code:\n{entry['code']}")
        if 'exec_result' in entry:
            er = entry['exec_result']
            print(f"OK: {er['ok']}  stdout: {er['stdout'].strip()[:100]}")
            if er['stderr']:
                print(f"stderr: {er['stderr'].strip()[:200]}")
else:
    print("Skipped — set HF_TOKEN to run LLM demos")

## 10. Side-by-Side: LLM vs Heuristic vs Deterministic

Compare all three agents on the same needle task.

In [ ]:
# Generate one task and run all agents on it
haystack, question, answer = generate_task(num_sentences=15, num_needles=1)
print(f"Question: {question}")
print(f"Expected: {answer}\n")

agents_to_compare = [
    DeterministicAgent(),
    HeuristicMultiStepAgent(),
]
if HF_TOKEN_SET:
    agents_to_compare.append(LLMAgent(max_steps=5))

print(f"{'Agent':<30} {'Prediction':<20} {'Correct':<10} {'Steps':<8}")
print("-" * 70)

for agent in agents_to_compare:
    import time as _time
    start = _time.time()
    predicted, transcript = agent.run_episode(haystack, question, answer)
    elapsed = _time.time() - start
    correct = predicted == answer
    print(f"{agent.name:<30} {predicted:<20} {str(correct):<10} {len(transcript):<8} ({elapsed:.3f}s)")

## 11. Multi-Step Retry Transcript

When the LLM generates code that fails (e.g., sandbox errors), it receives the error and retries. Here we show the full conversation transcript.

In [ ]:
if HF_TOKEN_SET:
    # Use a slightly harder task to increase chance of seeing a retry
    context, question, answer = generate_kv_extraction_task(num_entries=25)

    llm_agent = LLMAgent(max_steps=5)
    predicted, transcript = llm_agent.run_episode(context, question, answer)

    print(f"Question: {question}")
    print(f"Expected answer: {answer}")
    print(f"Predicted answer: {predicted}")
    print(f"Total steps: {len(transcript)}")
    print(f"Correct: {predicted == answer}")
    print()

    for entry in transcript:
        print(f"{'='*60}")
        print(f"STEP {entry['step']}: {entry['action']}")
        print(f"{'='*60}")
        if 'llm_response' in entry:
            print(f"LLM Response (preview):\n{entry['llm_response'][:300]}")
        if 'code' in entry:
            print(f"\nExtracted Code:\n{entry['code']}")
        if 'exec_result' in entry:
            er = entry['exec_result']
            print(f"\nExecution: ok={er['ok']}  time={er['runtime_sec']:.4f}s")
            if er['stdout']:
                print(f"stdout: {er['stdout'].strip()[:200]}")
            if er['stderr']:
                print(f"stderr: {er['stderr'].strip()[:200]}")
        if 'error' in entry:
            print(f"Error: {entry['error']}")
        print()
else:
    print("Skipped — set HF_TOKEN to run LLM demos")